> **통합 안내:** 이 노트북의 블록 생성은 `02_블록만들기.ipynb`의 8절로 옮겼습니다. 신고기준 블록 번호가 바뀌었습니다: 신고범위 b01 · 진단기준 b02 · 신고시기 b03 · 신고방법 b04 (이 노트북의 이전 번호 b04 진단기준은 더 이상 쓰지 않습니다). 기록용으로 남겨 둡니다.

# P3(남은 것). 법정 신고기준 블록 — 홍역 · 성홍열

## 이 노트북이 하는 일
- 진단·신고기준 변환본(237쪽)에서 **홍역(PDF 89쪽)** 과 **성홍열(PDF 104쪽)** 의 신고 범위·시기·방법을 블록으로 만듭니다.
- 원본 대조: PDF 이미지 `page-0089.png`, `page-0104.png`를 직접 보고 확인했습니다.
  - 홍역: 원본 **인쇄 81쪽**(PDF 89쪽). 신고범위 환자·의사환자 / 신고시기 24시간 이내 / 신고방법 팩스·방역통합정보시스템 → 변환본과 일치.
  - 성홍열: 원본 **인쇄 96쪽**(PDF 104쪽). 같은 내용 → 변환본과 일치.
- 블록 번호는 행 순서대로 b01(신고범위), b02(신고시기), b03(신고방법)입니다.
- 변환본은 자동 변환이라 **쪽 번호 기준은 PDF 순서**입니다. 인쇄 쪽은 `printed_page` 칸에 따로 적습니다.

In [1]:
import json
from preprocess.prevention_legal.pages import DOCS, excerpt_in_page

# 신고기준 문서의 문서 키와 document_id
KEY = "diagnosis"
DOC_ID = DOCS[KEY][0]
BLOCKS = "../../data/interim/prevention_legal/blocks.jsonl"

# 기존 블록 (중복 확인용)
existing = [json.loads(line) for line in open(BLOCKS, encoding="utf-8")]
existing_ids = {b["chunk_id"] for b in existing}
print("문서 ID:", DOC_ID, "| 기존 블록 수:", len(existing))

문서 ID: kdca_diagnosis_reporting_2026 | 기존 블록 수: 8


## 1. 블록 계획
(PDF 쪽, 블록 번호, 소제목, 인쇄 쪽, 확인할 구절, 블록 원문)을 적습니다. 구절은 `excerpt_in_page`로 원문 쪽에서 찾을 때 쓰는 짧은 문장입니다.

In [2]:
# 블록 계획: (PDF 쪽, 블록 번호, 소제목, 인쇄 쪽, 확인 구절, 블록 원문)
PLAN = [
    (89, 1, "21. [제2급-3] 홍역(Measles) · 신고범위", 81, "환자, 의사환자",
     "신고범위: 환자, 의사환자"),
    (89, 2, "21. [제2급-3] 홍역(Measles) · 신고시기", 81, "24시간 이내 신고",
     "신고시기: 24시간 이내 신고"),
    (89, 3, "21. [제2급-3] 홍역(Measles) · 신고방법", 81, "보건소로 팩스 또는 질병관리청 방역통합정보시스템",
     "신고방법: 신고서(발생 신고 및 사망(검안) 신고는 부록 1-1, 병원체검사결과 신고는 부록 1-2)를 작성하여 관할 보건소로 팩스 또는 질병관리청 방역통합정보시스템(https://eid.kdca.go.kr)으로 신고"),
    (104, 1, "36. [제2급-18] 성홍열(Scarlet fever) · 신고범위", 96, "환자, 의사환자",
     "신고범위: 환자, 의사환자"),
    (104, 2, "36. [제2급-18] 성홍열(Scarlet fever) · 신고시기", 96, "24시간 이내 신고",
     "신고시기: 24시간 이내 신고"),
    (104, 3, "36. [제2급-18] 성홍열(Scarlet fever) · 신고방법", 96, "보건소로 팩스 또는 질병관리청 방역통합정보시스템",
     "신고방법: 신고서(발생 신고 및 사망(검안) 신고는 부록 1-1, 병원체검사결과 신고는 부록 1-2)를 작성하여 관할 보건소로 팩스 또는 질병관리청 방역통합정보시스템(https://eid.kdca.go.kr)으로 신고"),
]
print("계획 블록:", len(PLAN), "개")

계획 블록: 6 개


## 2. 원문 확인
각 블록의 확인 구절이 해당 PDF 쪽 변환본에 있는지 봅니다. 실패하면 저장하지 않고 멈춥니다.

In [3]:
# 확인 구절이 해당 쪽에 없으면 여기서 멈춘다 (저장 전 점검)
new_blocks = []
for page, block_no, section, printed, phrase, text in PLAN:
    ok = excerpt_in_page(phrase, KEY, page)
    print(("원문 확인 " if ok else "확인 필요 "), f"PDF {page}쪽 b{block_no:02d} | {phrase[:30]}")
    assert ok, f"{page}쪽에서 구절을 찾지 못했습니다: {phrase}"
    new_blocks.append({
        "chunk_id": f"{DOC_ID}:p{page:04d}:b{block_no:02d}",
        "document_id": DOC_ID,
        "text": text,
        "pdf_page": page,
        "printed_page": printed,
        "section": section,
    })

원문 확인  PDF 89쪽 b01 | 환자, 의사환자
원문 확인  PDF 89쪽 b02 | 24시간 이내 신고
원문 확인  PDF 89쪽 b03 | 보건소로 팩스 또는 질병관리청 방역통합정보시스템
원문 확인  PDF 104쪽 b01 | 환자, 의사환자
원문 확인  PDF 104쪽 b02 | 24시간 이내 신고
원문 확인  PDF 104쪽 b03 | 보건소로 팩스 또는 질병관리청 방역통합정보시스템


## 3. 저장 (기존 블록은 지우지 않고, 같은 chunk_id 는 추가하지 않는다)

In [4]:
ids = [b["chunk_id"] for b in new_blocks]
assert len(ids) == len(set(ids)), "새 블록 안에서 chunk_id 가 중복됩니다."

to_add = [b for b in new_blocks if b["chunk_id"] not in existing_ids]
with open(BLOCKS, "a", encoding="utf-8") as f:
    for b in to_add:
        f.write(json.dumps(b, ensure_ascii=False) + chr(10))

print("추가 저장:", len(to_add), "개 | 이미 있어서 건너뜀:", len(new_blocks) - len(to_add))
print("전체 블록 수:", sum(1 for _ in open(BLOCKS, encoding="utf-8")))

추가 저장: 6 개 | 이미 있어서 건너뜀: 0
전체 블록 수: 14
